In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [27]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete. Also make sure to add the type of task (Python, JSON, or Regex) to the object.
        
        Can we add solution criteria for the best answer? For example, if the task is to write a Python function 
        that adds two numbers, the solution criteria could be that the function should take two arguments and 
        return their sum. If the task is to write a regular expression that matches email addresses, 
        the solution criteria could be that the regular expression should match valid email addresses and not 
        match invalid ones.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "python, json, or regex",
                "solution_criteria": "Description of solution criteria"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    
    response = chat(messages, stop_sequences=["```"])
    
    return json.loads(response)

In [28]:
dataset = generate_dataset()

# Write the dataset to a file
with open("prompt_eval_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [29]:
def run_prompt(test_case):
    """Merge the prompt and test case into a single prompt and run it through the model."""
    prompt = f"""
    Please solve the following task: 
    
    {test_case['task']}
    
    * Make sure to respond with only python code, a JSON object, or a regular expression depending on the format specified in the test case. 
    * Do not include any explanations or text, only the python, json, or plain text regex.
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    response = chat(messages, stop_sequences=["```"])
    print(response)
    return response

In [30]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
        You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

        Original Task:
        <task>
        {test_case["task"]}
        </task>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>
        
        Solution Criteria:
        <criteria>
        {test_case["solution_criteria"]}
        </criteria>

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [32]:
import re
import ast

def valdiate_json(text):
    """Validate that the provided text is a well-formed JSON object."""
    try:
        # Attempt to parse the text as JSON
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError as e:
        return 0
    
def validate_python(text):
    """Validate that the provided text is a well-formed Python function."""
    try:
        # Attempt to parse the text as Python code
        ast.parse(text.strip())
        return 10
    except SyntaxError as e:
        return 0
    
def validate_regex(text):
    """Validate that the provided text is a well-formed regular expression."""
    try:
        re.compile(text.strip())
        return 10
    except re.error as e:
        return 0
    
# Function to run the evaluation by format of the test case
def grade_syntax(response, test_case):
    """Grade the syntax of the response based on the format required by the test case."""
    format = test_case["format"]
    
    print(f"Grading syntax for format: {format}")
    print(f"Response to validate:\n{response}")
    
    if format == "python":
        return validate_python(response)
    elif format == "json":
        return valdiate_json(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError("Unknown task format")

In [33]:
def run_test_case(test_case):
    """Call run_prompt and grade the response."""
    response = run_prompt(test_case)
    
    # Score the response using the model
    evaluation = grade_by_model(test_case, response)
    grade = evaluation["score"]
    
    reasoning = evaluation["reasoning"]
    
    # Run syntax validation and average it with the model score
    syntax_score = grade_syntax(response, test_case)
    final_grade = (grade + syntax_score) / 2
    
    return {
        "response": response,
        "test_case": test_case,
        "grade": final_grade,
        "reasoning": reasoning
    }

In [34]:
from statistics import mean

def run_eval(dataset):
    """Load tehe dataset and call run_test_case on each test case."""
    results = []
    
    #iterate through the dataset and run each test case
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    # Calculate average score
    average_score = mean([result["grade"] for result in results])
    print(f"Average Score: {average_score}")

    return results

In [35]:
# Load the dataset from the file
with open("prompt_eval_dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)


import re

def parse_s3_uri(uri):
    """
    Parses an AWS S3 bucket URI and returns a dictionary with 'bucket' and 'key' fields.
    
    Args:
        uri (str): An S3 URI in the format s3://bucket-name/key/path
        
    Returns:
        dict: A dictionary with 'bucket' and 'key' fields
        
    Raises:
        ValueError: If the URI is not a valid S3 URI
    """
    pattern = r'^s3://([a-z0-9.-]+)/(.*?)$'
    match = re.match(pattern, uri)
    
    if not match:
        raise ValueError(f"Invalid S3 URI: {uri}")
    
    bucket = match.group(1)
    key = match.group(2)
    
    return {
        'bucket': bucket,
        'key': key
    }

Grading syntax for format: python
Response to validate:

import re

def parse_s3_uri(uri):
    """
    Parses an AWS S3 bucket URI and returns a dictionary with 'bucket' and 'key' fields.
    
    Args:
        uri (str): An S3 URI in the format s3://bucket-name/key/path
        
    Returns:
        dict: A dictionary with 'bucket' and 'k

In [36]:
print(json.dumps(results, indent=2))

[
  {
    "response": "\nimport re\n\ndef parse_s3_uri(uri):\n    \"\"\"\n    Parses an AWS S3 bucket URI and returns a dictionary with 'bucket' and 'key' fields.\n    \n    Args:\n        uri (str): An S3 URI in the format s3://bucket-name/key/path\n        \n    Returns:\n        dict: A dictionary with 'bucket' and 'key' fields\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n    \"\"\"\n    pattern = r'^s3://([a-z0-9.-]+)/(.*?)$'\n    match = re.match(pattern, uri)\n    \n    if not match:\n        raise ValueError(f\"Invalid S3 URI: {uri}\")\n    \n    bucket = match.group(1)\n    key = match.group(2)\n    \n    return {\n        'bucket': bucket,\n        'key': key\n    }\n",
    "test_case": {
      "task": "Write a Python function that parses an AWS S3 bucket URI (s3://bucket-name/key/path) and returns a dictionary with separate 'bucket' and 'key' fields",
      "format": "python",
      "solution_criteria": "The function should accept a single S3 